%md
# 01_ingest_bronze_alerts — Capa Bronze de señales TradingView

## 1. Objetivo de la capa Bronze

La capa **Bronze** ingiere los eventos crudos exportados desde AWS S3 hacia una tabla Delta en Databricks.

Esta capa representa el primer punto de entrada del Lakehouse. Su objetivo principal es conservar los datos con la menor transformación posible, manteniendo trazabilidad completa desde el archivo JSONL generado por Lambda hasta la tabla Bronze.

La fuente principal es:

    s3://trading-lakehouse-btc-s3/bronze/trading_alerts/

El destino principal es:

    trading.bronze.alerts_raw

---

## 2. Funcionalidades principales

La capa Bronze realiza las siguientes funciones:

1. Lee archivos JSONL desde AWS S3.
2. Usa Databricks Auto Loader para ingesta incremental.
3. Usa Managed File Events para detectar nuevos archivos.
4. Infiere el schema inicial de los eventos JSON.
5. Permite evolución de schema con `cloudFiles.schemaEvolutionMode = addNewColumns`.
6. Guarda el schema inferido en una ruta externa de S3.
7. Usa checkpoint externo para controlar qué archivos ya fueron procesados.
8. Añade metadatos técnicos de ingesta Databricks.
9. Conserva el payload exportado por Lambda sin aplanarlo todavía.
10. Escribe los datos en formato Delta.
11. Deja los eventos listos para ser transformados por la capa Silver.
12. Mantiene compatibilidad con Unity Catalog.
13. Evita depender de `input_file_name()` usando `_metadata.file_path`.
14. Procesa los datos disponibles con `availableNow=True`.
15. Permite reprocesos controlados mediante checkpoint.

---

## 3. Rol dentro del Lakehouse

Flujo actual:

    TradingView
      ↓
    Vercel /api/webhook
      ↓
    AWS SQS
      ↓
    AWS Lambda Consumer
      ↓
    Supabase operacional
      ↓
    AWS S3 Bronze JSONL
      ↓
    01_ingest_bronze_alerts
      ↓
    trading.bronze.alerts_raw
      ↓
    02_transform_silver_alerts
      ↓
    trading.silver.alerts_clean
      ↓
    03_build_gold_tables
      ↓
    trading.gold.validated_trade_signals
    trading.gold.signal_quality_metrics
    trading.gold.ml_training_candidates

La capa Bronze es el punto donde Databricks comienza a gobernar los datos analíticos dentro del Lakehouse.

---

## 4. Fuente de datos

La fuente de Bronze es el bucket S3 donde Lambda exporta los eventos procesados:

    s3://trading-lakehouse-btc-s3/bronze/trading_alerts/

La estructura particionada esperada es:

    bronze/trading_alerts/
      processing_date=YYYY-MM-DD/
        symbol=btcusdc/
          tf=1m/
            message_type=logical_event_full/
              event_<event_uid>_<timestamp>.jsonl

Ejemplo:

    s3://trading-lakehouse-btc-s3/bronze/trading_alerts/processing_date=2026-05-02/symbol=btcusdc/tf=1m/message_type=logical_event_full/event_test_s3_final_010_20260502T211920389832.jsonl

---

## 5. Tabla destino

La tabla destino de Bronze es:

    trading.bronze.alerts_raw

Esta tabla contiene los eventos exportados por el backend en formato estructurado/semi-estructurado, manteniendo los bloques principales:

    event_uid
    schema_version
    message_type
    trace_id
    symbol
    tf
    event
    side
    price
    validation_approve
    validation_confidence
    probability_tp_before_sl
    raw_payload
    raw_validation
    ingestion_ts
    processing_date
    source
    lakehouse_layer

Además, Databricks añade campos técnicos de ingesta:

    _databricks_ingestion_ts
    _source_file
    _source_file_modification_time
    _processing_date_dbx

---

## 6. Contrato Bronze exportado por Lambda

Lambda construye un evento Bronze con el siguiente contrato lógico:

    event_uid
    schema_version
    message_type
    trace_id
    symbol
    tf
    event
    side
    price
    validation_approve
    validation_confidence
    probability_tp_before_sl
    raw_payload
    raw_validation
    ingestion_ts
    processing_date
    source
    lakehouse_layer

Donde:

    raw_payload      = payload completo recibido y/o ensamblado desde TradingView
    raw_validation   = resultado completo de run_validation()
    ingestion_ts     = timestamp de exportación desde backend/Lambda
    processing_date  = fecha usada como partición lógica
    lakehouse_layer  = bronze

---

## 7. Auto Loader

La capa Bronze usa Databricks Auto Loader para leer archivos desde S3 de forma incremental.

Auto Loader permite:

1. Leer archivos nuevos sin reprocesar todo el directorio.
2. Mantener estado de archivos procesados.
3. Gestionar evolución de schema.
4. Integrarse con file events.
5. Escalar mejor que una lectura manual de carpetas.
6. Automatizar la ingesta desde almacenamiento cloud hacia Delta Lake.

---

## 8. Managed File Events

La capa Bronze usa Managed File Events para detectar nuevos archivos en la external location.

Ventajas:

1. Reduce la necesidad de listar continuamente el bucket S3.
2. Mejora la escalabilidad.
3. Mejora la detección de nuevos archivos.
4. Se integra con Unity Catalog y external locations.
5. Facilita una ingesta más eficiente desde S3.

Estado actual:

    VALIDADO:
    - External location creada.
    - File events funcionando.
    - Auto Loader puede leer desde S3.
    - `_metadata.file_path` funciona correctamente en Unity Catalog.

---

## 9. Schema location

Auto Loader guarda la evolución del schema en:

    s3://trading-lakehouse-btc-s3/schemas/bronze/trading_alerts/

Esta ruta permite que Databricks recuerde el schema inferido y lo actualice si aparecen nuevas columnas.

Configuración conceptual:

    schemaLocation
    inferColumnTypes
    schemaEvolutionMode = addNewColumns

Esto permite que futuros campos enviados por TradingView, Lambda o `validation_service` puedan incorporarse sin romper la ingesta Bronze.

---

## 10. Checkpoint

La capa Bronze usa checkpoint en:

    s3://trading-lakehouse-btc-s3/checkpoints/bronze/trading_alerts/

El checkpoint guarda el estado del stream:

1. Qué archivos ya fueron procesados.
2. Progreso del stream.
3. Información necesaria para evitar duplicados.
4. Estado de ejecución de Auto Loader.

Regla importante:

    No borrar el checkpoint salvo que se quiera reprocesar todo de forma controlada.

Si se borra el checkpoint, Auto Loader puede volver a leer archivos ya procesados y generar duplicados en Bronze.

---

## 11. Modo de ejecución

El notebook usa:

    trigger(availableNow=True)

Esto significa:

    Procesa todos los archivos disponibles ahora
    ↓
    Escribe en la tabla Delta Bronze
    ↓
    Se detiene automáticamente

Este modo es adecuado para:

1. Ejecuciones manuales.
2. Jobs programados.
3. Procesamiento incremental por lotes.
4. Backfills controlados.
5. Integración con pipelines Databricks.

---

## 12. Metadatos técnicos añadidos

La capa Bronze añade campos técnicos:

    _databricks_ingestion_ts
    _source_file
    _source_file_modification_time
    _processing_date_dbx

Descripción:

    _databricks_ingestion_ts
        Timestamp de cuándo Databricks ingirió el registro.

    _source_file
        Ruta completa del archivo S3 leído por Auto Loader.

    _source_file_modification_time
        Timestamp de modificación del archivo fuente.

    _processing_date_dbx
        Fecha de procesamiento calculada por Databricks.

Estos campos permiten trazar cualquier fila de Bronze hasta el archivo físico S3 original.

---

## 13. Trazabilidad

Bronze permite reconstruir el recorrido completo de una señal:

    event_uid
      ↓
    raw_payload
      ↓
    raw_validation
      ↓
    _source_file
      ↓
    S3 JSONL original
      ↓
    Lambda trace_id
      ↓
    Supabase operational records

Esto es importante para:

1. Auditoría.
2. Debugging.
3. Backfills.
4. Reprocesamiento.
5. Evolución del schema.
6. Validación de consistencia entre Supabase y S3.
7. Construcción de datasets futuros.

---

## 14. Relación con Supabase

Supabase y Bronze tienen roles distintos.

Supabase:

    - Operación casi real-time.
    - Dashboard operativo.
    - Estado actual de setups.
    - Validaciones recientes.
    - Logs de errores.
    - Buffer temporal CORE/EXTRA.

S3 + Bronze:

    - Histórico analítico.
    - Persistencia masiva.
    - Dataset para ML.
    - Backtesting.
    - Trazabilidad completa.
    - Entrada del Lakehouse.

Bronze no reemplaza Supabase. Bronze es la base analítica del Lakehouse.

---

## 15. Relación con Silver

Bronze conserva datos crudos y semi-estructurados.

Silver transforma esos datos en columnas limpias y consultables.

Flujo:

    trading.bronze.alerts_raw
      ↓
    trading.silver.alerts_clean

Bronze mantiene:

    raw_payload
    raw_validation
    metadatos de archivo
    estructura original exportada

Silver aplana:

    validation
    trade plan
    market_snapshot
    structure_snapshot
    alert_reused
    ml_tracking

---

## 16. Relación con Gold

Gold consume Silver, no Bronze directamente.

Flujo:

    Bronze = ingesta raw
    Silver = limpieza y aplanamiento
    Gold = consumo final

Las tablas Gold construidas a partir de Silver son:

    trading.gold.validated_trade_signals
    trading.gold.signal_quality_metrics
    trading.gold.ml_training_candidates

---

## 17. Estado actual

VALIDADO:

- Lambda exporta correctamente eventos JSONL a S3.
- La ruta S3 Bronze existe y recibe archivos.
- Databricks puede leer desde S3.
- Auto Loader ingiere archivos desde `RAW_PATH`.
- Managed File Events funciona.
- Unity Catalog external location funciona.
- `_metadata.file_path` funciona correctamente.
- La tabla `trading.bronze.alerts_raw` recibe eventos.
- Bronze alimenta correctamente la capa Silver.

PENDIENTE:

- Programar ejecución automática Bronze → Silver → Gold.
- Definir política de retención de checkpoints.
- Definir política de compactación/OPTIMIZE.
- Definir estrategia de backfill.
- Definir tabla de control de calidad de ingesta.
- Implementar monitoreo de errores de Auto Loader.

---

## 18. Riesgos y consideraciones

### Checkpoint

No borrar el checkpoint salvo para reprocesos controlados.

    s3://trading-lakehouse-btc-s3/checkpoints/bronze/trading_alerts/

### Schema evolution

Si aparecen nuevos campos en los JSONL, Auto Loader puede añadir columnas nuevas gracias a la evolución de schema.

### Duplicados

Bronze puede contener duplicados si se reprocesan archivos manualmente o se borra el checkpoint.

La deduplicación principal se realiza en Silver mediante:

    event_unique_key

### Coste

El coste principal puede venir de:

    - lecturas de S3
    - ejecución del cluster Databricks
    - almacenamiento Delta
    - metadatos/checkpoints/schema location

### Reprocesamiento

Para reprocesar todo desde cero habría que decidir cuidadosamente si:

    - borrar tabla Bronze
    - borrar checkpoint
    - conservar schema location
    - reconstruir Silver y Gold

---

## 19. Resumen ejecutivo

La capa `01_ingest_bronze_alerts` es la puerta de entrada del Lakehouse.

Toma los eventos JSONL generados por Lambda en S3 y los ingiere en Databricks mediante Auto Loader, conservando el evento completo, la validación completa y los metadatos técnicos del archivo fuente.

La tabla resultante:

    trading.bronze.alerts_raw

queda preparada como fuente para:

- transformación Silver,
- auditoría,
- trazabilidad,
- reprocesamiento,
- evolución de schema,
- backtesting,
- Machine Learning,
- construcción de tablas Gold.

In [0]:
RAW_PATH = "s3://trading-lakehouse-btc-s3/bronze/trading_alerts/"

display(dbutils.fs.ls(RAW_PATH))

In [0]:
df_test = spark.read.json(RAW_PATH)

display(df_test.limit(10))
df_test.printSchema()

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS trading;

CREATE SCHEMA IF NOT EXISTS trading.bronze;
CREATE SCHEMA IF NOT EXISTS trading.silver;
CREATE SCHEMA IF NOT EXISTS trading.gold;

In [0]:
# ============================================================
# 01_ingest_bronze_alerts
# Rebuild Bronze desde S3 usando Auto Loader
# Compatible con evolución de schema
# ============================================================

from pyspark.sql import functions as F

RAW_PATH = "s3://trading-lakehouse-btc-s3/bronze/trading_alerts/"
SCHEMA_PATH = "s3://trading-lakehouse-btc-s3/schemas/bronze/trading_alerts/"
CHECKPOINT_PATH = "s3://trading-lakehouse-btc-s3/checkpoints/bronze/trading_alerts/"

TARGET_TABLE = "trading.bronze.alerts_raw"

spark.sql("CREATE SCHEMA IF NOT EXISTS trading.bronze")

# ------------------------------------------------------------
# Rebuild limpio: borrar tabla, checkpoint y schema Auto Loader
# ------------------------------------------------------------

spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")

for path_name, path_value in {
    "CHECKPOINT_PATH": CHECKPOINT_PATH,
    "SCHEMA_PATH": SCHEMA_PATH,
}.items():
    try:
        dbutils.fs.rm(path_value, recurse=True)
        print(f"Removed {path_name}: {path_value}")
    except Exception as e:
        print(f"Remove skipped or failed for {path_name}: {path_value} -> {str(e)}")

# ------------------------------------------------------------
# Read Bronze desde S3 con Auto Loader
# ------------------------------------------------------------

df_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_PATH)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("rescuedDataColumn", "_rescued_data")
    .load(RAW_PATH)
)

df_bronze = (
    df_raw
    .withColumn("_databricks_ingestion_ts", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_source_file_modification_time", F.col("_metadata.file_modification_time"))
    .withColumn("_processing_date_dbx", F.to_date(F.current_timestamp()))
)

# ------------------------------------------------------------
# Write Bronze Delta
# ------------------------------------------------------------

query = (
    df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
)

query.awaitTermination()

print(f"Bronze ingestion completed: {TARGET_TABLE}")

In [0]:
from pyspark.sql import functions as F

df = spark.table("trading.bronze.alerts_raw")

df.select(
    "event_uid",
    "message_type",
    F.col("raw_payload.signal.bar_time").alias("raw_payload_signal_bar_time"),
    F.col("raw_payload.signal.timestamp").alias("raw_payload_signal_timestamp"),
    F.col("raw_payload.signal.price").alias("raw_payload_signal_price")
).where(
    F.col("message_type") == "logical_event_full"
).show(20, truncate=False)

### VALIDACION

In [0]:
spark.catalog.tableExists("trading.bronze.alerts_raw")

In [0]:
%sql
-- ============================================================
-- SQL DE VALIDACIÓN — CAPA BRONZE
-- Tabla: trading.bronze.alerts_raw
-- ============================================================


-- ============================================================
-- 1. Conteo total de registros Bronze
-- ============================================================

SELECT
  COUNT(*) AS total_rows
FROM trading.bronze.alerts_raw;


-- ============================================================
-- 2. Últimos eventos ingeridos en Bronze
-- ============================================================

SELECT
  event_uid,
  schema_version,
  message_type,
  trace_id,
  symbol,
  tf,
  event,
  side,
  price,
  validation_approve,
  validation_confidence,
  probability_tp_before_sl,
  ingestion_ts,
  processing_date,
  source,
  lakehouse_layer,
  _databricks_ingestion_ts,
  _source_file
FROM trading.bronze.alerts_raw
ORDER BY _databricks_ingestion_ts DESC
LIMIT 20;


-- ============================================================
-- 3. Buscar evento específico por event_uid
-- Cambia el valor por el event_uid que quieras validar
-- ============================================================

SELECT
  *
FROM trading.bronze.alerts_raw
WHERE event_uid = 'test_s3_final_010';


-- ============================================================
-- 4. Validar eventos aprobados
-- ============================================================

SELECT
  event_uid,
  message_type,
  symbol,
  tf,
  event,
  side,
  validation_approve,
  validation_confidence,
  probability_tp_before_sl,
  ingestion_ts,
  _databricks_ingestion_ts,
  _source_file
FROM trading.bronze.alerts_raw
WHERE validation_approve = true
ORDER BY _databricks_ingestion_ts DESC
LIMIT 50;


-- ============================================================
-- 5. Validar eventos rechazados
-- ============================================================

SELECT
  event_uid,
  message_type,
  symbol,
  tf,
  event,
  side,
  validation_approve,
  validation_confidence,
  probability_tp_before_sl,
  ingestion_ts,
  _databricks_ingestion_ts,
  _source_file
FROM trading.bronze.alerts_raw
WHERE validation_approve = false
ORDER BY _databricks_ingestion_ts DESC
LIMIT 50;


-- ============================================================
-- 6. Conteo por message_type
-- ============================================================

SELECT
  message_type,
  COUNT(*) AS total_rows
FROM trading.bronze.alerts_raw
GROUP BY message_type
ORDER BY total_rows DESC;


-- ============================================================
-- 7. Conteo por evento
-- ============================================================

SELECT
  event,
  COUNT(*) AS total_rows
FROM trading.bronze.alerts_raw
GROUP BY event
ORDER BY total_rows DESC;


-- ============================================================
-- 8. Conteo por símbolo, timeframe y evento
-- ============================================================

SELECT
  symbol,
  tf,
  event,
  side,
  COUNT(*) AS total_rows
FROM trading.bronze.alerts_raw
GROUP BY symbol, tf, event, side
ORDER BY total_rows DESC;


-- ============================================================
-- 9. Conteo por fecha de procesamiento
-- ============================================================

SELECT
  processing_date,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT event_uid) AS distinct_events
FROM trading.bronze.alerts_raw
GROUP BY processing_date
ORDER BY processing_date DESC;


-- ============================================================
-- 10. Validar archivos fuente procesados
-- ============================================================

SELECT
  _source_file,
  COUNT(*) AS rows_in_file,
  MIN(_databricks_ingestion_ts) AS first_ingested_at,
  MAX(_databricks_ingestion_ts) AS last_ingested_at
FROM trading.bronze.alerts_raw
GROUP BY _source_file
ORDER BY last_ingested_at DESC;


-- ============================================================
-- 11. Detectar posibles duplicados por event_uid
-- ============================================================

SELECT
  event_uid,
  COUNT(*) AS rows_count,
  COUNT(DISTINCT _source_file) AS source_files_count,
  MIN(_databricks_ingestion_ts) AS first_ingested_at,
  MAX(_databricks_ingestion_ts) AS last_ingested_at
FROM trading.bronze.alerts_raw
WHERE event_uid IS NOT NULL
GROUP BY event_uid
HAVING COUNT(*) > 1
ORDER BY rows_count DESC, last_ingested_at DESC;


-- ============================================================
-- 12. Detectar duplicados por event_uid + message_type + event
-- Esta es la combinación que luego Silver usa para deduplicar
-- ============================================================

SELECT
  event_uid,
  message_type,
  event,
  COUNT(*) AS rows_count,
  COUNT(DISTINCT _source_file) AS source_files_count,
  MIN(_databricks_ingestion_ts) AS first_ingested_at,
  MAX(_databricks_ingestion_ts) AS last_ingested_at
FROM trading.bronze.alerts_raw
WHERE event_uid IS NOT NULL
GROUP BY event_uid, message_type, event
HAVING COUNT(*) > 1
ORDER BY rows_count DESC, last_ingested_at DESC;


-- ============================================================
-- 13. Validar registros con event_uid nulo
-- ============================================================

SELECT
  *
FROM trading.bronze.alerts_raw
WHERE event_uid IS NULL
ORDER BY _databricks_ingestion_ts DESC
LIMIT 50;


-- ============================================================
-- 14. Validar registros con raw_payload nulo
-- ============================================================

SELECT
  event_uid,
  message_type,
  symbol,
  tf,
  event,
  side,
  ingestion_ts,
  _source_file
FROM trading.bronze.alerts_raw
WHERE raw_payload IS NULL
ORDER BY _databricks_ingestion_ts DESC
LIMIT 50;


-- ============================================================
-- 15. Validar registros con raw_validation nulo
-- ============================================================

SELECT
  event_uid,
  message_type,
  symbol,
  tf,
  event,
  side,
  validation_approve,
  validation_confidence,
  probability_tp_before_sl,
  ingestion_ts,
  _source_file
FROM trading.bronze.alerts_raw
WHERE raw_validation IS NULL
ORDER BY _databricks_ingestion_ts DESC
LIMIT 50;


-- ============================================================
-- 16. Validar consistencia entre columnas top-level y raw_payload.signal
-- ============================================================

SELECT
  event_uid,
  symbol AS top_symbol,
  raw_payload.signal.symbol AS payload_symbol,
  tf AS top_tf,
  raw_payload.signal.tf AS payload_tf,
  event AS top_event,
  raw_payload.signal.event AS payload_event,
  side AS top_side,
  raw_payload.signal.side AS payload_side,
  _source_file
FROM trading.bronze.alerts_raw
WHERE
  symbol <> raw_payload.signal.symbol
  OR tf <> raw_payload.signal.tf
  OR event <> raw_payload.signal.event
  OR side <> raw_payload.signal.side
ORDER BY _databricks_ingestion_ts DESC
LIMIT 50;


-- ============================================================
-- 17. Validar consistencia entre columnas top-level y raw_validation.validation
-- ============================================================

SELECT
  event_uid,
  validation_approve AS top_validation_approve,
  raw_validation.validation.approve AS raw_validation_approve,
  validation_confidence AS top_validation_confidence,
  raw_validation.validation.confidence AS raw_validation_confidence,
  probability_tp_before_sl AS top_probability_tp_before_sl,
  raw_validation.validation.probability_tp_before_sl AS raw_probability_tp_before_sl,
  _source_file
FROM trading.bronze.alerts_raw
WHERE
  validation_approve <> raw_validation.validation.approve
  OR validation_confidence <> raw_validation.validation.confidence
  OR probability_tp_before_sl <> raw_validation.validation.probability_tp_before_sl
ORDER BY _databricks_ingestion_ts DESC
LIMIT 50;


-- ============================================================
-- 18. Validar distribución de aprobaciones
-- ============================================================

SELECT
  validation_approve,
  COUNT(*) AS total_rows,
  ROUND(AVG(validation_confidence), 2) AS avg_confidence,
  ROUND(AVG(probability_tp_before_sl), 4) AS avg_probability_tp_before_sl
FROM trading.bronze.alerts_raw
GROUP BY validation_approve
ORDER BY validation_approve DESC;


-- ============================================================
-- 19. Validar rango de probabilidad TP antes de SL
-- ============================================================

SELECT
  MIN(probability_tp_before_sl) AS min_probability,
  MAX(probability_tp_before_sl) AS max_probability,
  ROUND(AVG(probability_tp_before_sl), 4) AS avg_probability
FROM trading.bronze.alerts_raw
WHERE probability_tp_before_sl IS NOT NULL;


-- ============================================================
-- 20. Validar últimos archivos S3 por fecha de procesamiento
-- ============================================================

SELECT
  processing_date,
  symbol,
  tf,
  message_type,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT _source_file) AS total_files,
  MAX(_databricks_ingestion_ts) AS last_ingested_at
FROM trading.bronze.alerts_raw
GROUP BY processing_date, symbol, tf, message_type
ORDER BY processing_date DESC, last_ingested_at DESC;